# **Step 1 : Dependencies Installation**

In [1]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.5 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


# **Step 2 : Model_Evalution** 

In [2]:
from ultralytics import YOLO

# Load model (explicit task)
model = YOLO(
    "/kaggle/input/5-epoch/pytorch/default/1/best.pt",
    task="detect"
)

# Run validation
metrics = model.val(
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    batch=16,
    device=0,          # single GPU only
    split="val",
    save_json=True,
    plots=True,
    verbose=True,
    visualize=True     # optional (slow)
)

# Print metrics
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Per-class mAP:", metrics.box.maps)

# Confusion matrix as DataFrame
df_cm = metrics.confusion_matrix.to_df()
print(df_cm)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.251 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,848,445 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 64.6±29.4 MB/s, size: 459.2 KB)
val: Scanning /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val/labels... 4196 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4196/4196 222.3it/s 18.9s<0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%

# **Step 3 : CSV File Creation**

In [3]:
import os

# Convert metrics to CSV string
val_csv = metrics.to_csv()
print(val_csv)

# Define output path
dir_path = "/kaggle/working/runs/detect"
csv_filename = "validation_results.csv"

# Ensure directory exists
os.makedirs(dir_path, exist_ok=True)

# Full file path
full_path = os.path.join(dir_path, csv_filename)

# Write CSV
with open(full_path, "w") as f:
    f.write(val_csv)

print("Saved at:", full_path)


Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
animal,219,753,0.89557,0.01142,0.02256,0.06249,0.02469
autorickshaw,1436,3205,0.7144,0.42278,0.5312,0.49131,0.31353
bicycle,264,301,0.69242,0.0225,0.04358,0.06882,0.03104
bus,1061,1794,0.25509,0.53344,0.34514,0.32961,0.21816
car,2582,8793,0.59274,0.50608,0.546,0.50388,0.32845
caravan,18,18,1.0,0.0,0.0,0.01431,0.01194
motorcycle,2778,10059,0.58841,0.5169,0.55034,0.51298,0.26101
person,2105,8863,0.65593,0.26368,0.37615,0.3172,0.15401
rider,2454,9444,0.58105,0.39137,0.46771,0.41104,0.20756
traffic light,157,370,0.13805,0.06757,0.09073,0.03572,0.0143
traffic sign,774,1404,0.75896,0.09117,0.16278,0.13909,0.06512
train,4,4,0.0,0.0,0.0,0.0,0.0
truck,1564,2758,0.67205,0.21936,0.33076,0.3336,0.19554
vehicle fallback,1116,2072,0.58816,0.00208,0.00415,0.02921,0.0127

Saved at: /kaggle/working/runs/detect/validation_results.csv


# **Step 4 : Model BenchMarking**

In [4]:
from ultralytics.utils.benchmarks import benchmark

benchmark(
    model="/kaggle/input/5-epoch/pytorch/default/1/best.pt",
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    half=False,
    device=0,
)


Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6636.8/8062.4 GB disk)

Benchmarks complete for /kaggle/input/5-epoch/pytorch/default/1/best.pt on /kaggle/input/kaggle-yaml/kaggle_new_data.yaml at imgsz=640 (178.44s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+----------------------------------------------------------------------------------------------------------+
|      Format                  Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS   |
+==========================================================================================================+
| 1    PyTorch                 ✅         49.6        0.1313                14.59                    68.54 |
| 2    TorchScript             ❌         0.0         -                     -                        -     |
| 3    ONNX                    ❌         0.0         -                     -                        -     |
| 4    OpenVINO                ❌         0.0     

,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""PyTorch""","""✅""","""49.6""","""0.1313""","""14.59""","""68.54"""
"""2""","""TorchScript""","""❌""","""0.0""","""-""","""-""","""-"""
"""3""","""ONNX""","""❌""","""0.0""","""-""","""-""","""-"""
"""4""","""OpenVINO""","""❌""","""0.0""","""-""","""-""","""-"""
"""5""","""TensorRT""","""❌""","""0.0""","""-""","""-""","""-"""
"""6""","""CoreML""","""❌""","""0.0""","""-""","""-""","""-"""
"""7""","""TensorFlow SavedModel""","""❌""","""0.0""","""-""","""-""","""-"""
"""8""","""TensorFlow GraphDef""","""❌""","""0.0""","""-""","""-""","""-"""
"""9""","""TensorFlow Lite""","""❌""","""0.0""","""-""","""-""","""-"""
"""10""","""TensorFlow Edge TPU""","""❌""","""0.0""","""-""","""-""","""-"""


# **Step 5 : Zip File Creation** 

In [5]:
import shutil
import os

folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/evaluation_results"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")

ZIP file created at: /kaggle/working/evaluation_results.zip


In [6]:
from IPython.display import FileLink

FileLink("/kaggle/working/evaluation_results.zip")


/kaggle/working/evaluation_results.zip

In [7]:
import os

file_path = "/kaggle/working/evaluation_results.zip"
print("Exists:", os.path.isfile(file_path))
print("Size (MB):", os.path.getsize(file_path) / (1024*1024))


Exists: True
Size (MB): 709.554934501648
